In [1]:
#%pip install paramiko

In [2]:
import os
import posixpath
import time
from pathlib import Path

import pandas as pd
import paramiko
from dotenv import load_dotenv
from IPython import get_ipython
from selenium import webdriver
from selenium.common.exceptions import (
    ElementNotInteractableException,
    InvalidElementStateException,
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager


In [3]:
load_dotenv(dotenv_path=Path('.env'), override=True)

sftp_host = os.getenv('SFTP_HOST')
sftp_login = os.getenv('SFTP_LOGIN')
sftp_password = os.getenv('SFTP_PASSWORD')
sftp_base_folder = os.getenv('SFTP_FOLDER')
tmdb_login = os.getenv('TMDB_LOGIN')
tmdb_password = os.getenv('TMDB_PASSWORD')

if not sftp_host or not sftp_login or sftp_password is None or not sftp_base_folder:
    raise ValueError('Missing SFTP configuration in .env: SFTP_HOST, SFTP_LOGIN, SFTP_PASSWORD, SFTP_FOLDER')

if not tmdb_login or not tmdb_password:
    raise ValueError('Missing TMDB configuration in .env: TMDB_LOGIN, TMDB_PASSWORD')


In [4]:
DATASETS = [
    {
        'name': 'movies',
        'remote_folder': 'wikidata-id-movie-fix',
        'local_folder': 'wikidata-id-movie-fix',
        'local_file_basename': 'wikidata-id-movie-fix',
        'entity_path': 'movie',
        'id_column': 'ID_MOVIE',
        'erase_column': 'ID_MOVIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngmovieidstart',
        'missing_entity_message': 'Colonne ID_MOVIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for movie {lngid}. Skipping.'
    },
    {
        'name': 'series',
        'remote_folder': 'wikidata-id-serie-fix',
        'local_folder': 'wikidata-id-serie-fix',
        'local_file_basename': 'wikidata-id-serie-fix',
        'entity_path': 'tv',
        'id_column': 'ID_SERIE',
        'erase_column': 'ID_SERIE_ERASE_WIKIDATA_ID',
        'store_key': 'lngserieidstart',
        'missing_entity_message': 'Colonne ID_SERIE manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for serie {lngid}. Skipping.'
    },
    {
        'name': 'persons',
        'remote_folder': 'wikidata-id-person-fix',
        'local_folder': 'wikidata-id-person-fix',
        'local_file_basename': 'wikidata-id-person-fix',
        'entity_path': 'person',
        'id_column': 'ID_PERSON',
        'erase_column': 'ID_PERSON_ERASE_WIKIDATA_ID',
        'store_key': 'lngpersonidstart',
        'missing_entity_message': 'Colonne ID_PERSON manquante dans le fichier CSV',
        'missing_wikidata_message': 'Colonne ID_WIKIDATA manquante dans le fichier CSV',
        'timeout_message': 'wikidata_id field not found for person {lngid}. Skipping.',
        'intgoingdown': True
    }
]


In [5]:
SAVE_BUTTON_XPATH = '//input[@value="Save"]'


def download_new_csv_files(remote_folder, local_folder):
    remote_path = posixpath.join(sftp_base_folder, remote_folder)
    local_path = Path('./data') / local_folder
    local_path.mkdir(parents=True, exist_ok=True)

    downloaded_count = 0
    skipped_count = 0

    transport = paramiko.Transport((sftp_host, 22))
    transport.connect(username=sftp_login, password=sftp_password)
    sftp = paramiko.SFTPClient.from_transport(transport)

    try:
        for entry in sftp.listdir_attr(remote_path):
            if entry.filename in ('.', '..'):
                continue

            remote_file = posixpath.join(remote_path, entry.filename)
            local_file = local_path / entry.filename

            try:
                if (entry.st_mode & 0o170000) == 0o040000:
                    continue
            except Exception:
                pass

            if local_file.exists():
                skipped_count += 1
                continue

            sftp.get(remote_file, str(local_file))
            downloaded_count += 1
    finally:
        sftp.close()
        transport.close()

    print(f'SFTP folder: {remote_path}')
    print(f'Local folder: {local_path.resolve()}')
    print(f'Downloaded {downloaded_count} new file(s), skipped {skipped_count} existing file(s)')


def get_latest_csv(local_folder, local_file_basename):
    base_candidates = [Path('./data') / local_folder, Path('./data')]

    csv_candidates = []
    for base in base_candidates:
        if base.exists():
            csv_candidates.extend(base.glob(local_file_basename + '*.csv'))

    if not csv_candidates:
        raise FileNotFoundError(
            'No file found matching ' + local_file_basename + '*.csv in ./data or ./data/<local_folder>'
        )

    latest_csv = max(csv_candidates, key=lambda p: p.stat().st_mtime)
    print(f'Using latest CSV: {latest_csv}')
    return latest_csv


def set_store_value(store_key, value):
    ip = get_ipython()
    if ip is None:
        return

    ip.user_ns[store_key] = value
    ip.run_line_magic('store', store_key)


def init_driver():
    return webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))


def login_tmdb(driver):
    driver.get('https://www.themoviedb.org/login')
    time.sleep(10)

    username_field = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, 'username'))
    )
    username_field.clear()
    username_field.send_keys(tmdb_login)

    password_field = driver.find_element(By.NAME, 'password')
    password_field.clear()
    password_field.send_keys(tmdb_password)
    password_field.send_keys('\n')


def get_log_file_path():
    log_dir = Path('./data')
    log_dir.mkdir(parents=True, exist_ok=True)
    return log_dir / 'selenium-tmdb-wikidata_id.log'


def append_processed_log(content_type, record_id, wikidata_id):
    log_file_path = get_log_file_path()
    with log_file_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'{content_type};{record_id};{wikidata_id}\n')


def is_page_not_found(driver):
    elements = driver.find_elements(By.XPATH, "//h2[text()=\"Oops! We can't find the page you're looking for\"]")
    return len(elements) > 0


def is_logged_out(driver):
    return len(driver.find_elements(By.NAME, 'password')) > 0


def is_wikidata_id_locked(driver):
    # Every field carries a #wikidata_id_status padlock control: the discriminating
    # class is 'locked', never 'locked_status' alone.
    css_selectors = (
        'span#wikidata_id_status.glyphicons_v2.locked.locked_status',
        'span#wikidata_id_status.locked',
    )
    for css_selector in css_selectors:
        if len(driver.find_elements(By.CSS_SELECTOR, css_selector)) > 0:
            return True
    return False


def find_visible_element(driver, by, value):
    for element in driver.find_elements(by, value):
        try:
            if element.is_displayed():
                return element
        except StaleElementReferenceException:
            continue
    return None


def activate_external_ids_panel(driver):
    """The edit page keeps every section in the DOM and only shows the active one.

    When the active_nav_item URL parameter does not take effect, click the
    'External IDs' nav link so the wikidata_id input becomes interactable.
    """
    link = find_visible_element(driver, By.CSS_SELECTOR, 'a[href*="active_nav_item=external_ids"]')
    if link is None:
        return False

    driver.execute_script('arguments[0].click();', link)
    return True


def get_wikidata_id_field(driver, lngid, timeout_message):
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'wikidata_id'))
        )
    except TimeoutException:
        print(timeout_message.format(lngid=lngid))
        return None

    # The panel is rendered asynchronously and several sections may each hold a
    # wikidata_id input: keep the visible one.
    try:
        field = WebDriverWait(driver, 5).until(
            lambda d: find_visible_element(d, By.NAME, 'wikidata_id')
        )
    except TimeoutException:
        field = None

    if field is None and activate_external_ids_panel(driver):
        print("Panel 'External IDs' activated manually.")
        try:
            field = WebDriverWait(driver, 10).until(
                lambda d: find_visible_element(d, By.NAME, 'wikidata_id')
            )
        except TimeoutException:
            field = None

    if field is None:
        # Fall back to the first match so the caller can report its actual state.
        field = driver.find_element(By.NAME, 'wikidata_id')

    return field


def describe_field_state(field):
    try:
        return (
            f"displayed={field.is_displayed()} enabled={field.is_enabled()} "
            f"readonly={field.get_attribute('readonly')}"
        )
    except StaleElementReferenceException:
        return 'stale element'


def is_field_writable(field):
    try:
        return field.is_displayed() and field.is_enabled() and field.get_attribute('readonly') is None
    except StaleElementReferenceException:
        return False


def write_wikidata_id(field, strwikidataid=None):
    """Clear the field, then type strwikidataid when provided.

    Returns False (instead of raising) when TMDB refuses the edit, so the batch
    loop can skip the record and carry on.
    """
    try:
        field.clear()
        if strwikidataid is not None:
            field.send_keys(strwikidataid)
        return True
    except (
        InvalidElementStateException,
        ElementNotInteractableException,
        StaleElementReferenceException,
    ) as exc:
        print(f'wikidata_id field could not be edited ({type(exc).__name__}).')
        return False


def confirm_wikidata_id(driver, expected_value):
    """Re-read the field after Save so only real writes are reported and logged."""
    def has_expected_value(d):
        field = find_visible_element(d, By.NAME, 'wikidata_id')
        if field is None:
            return False
        try:
            return field.get_attribute('value') == expected_value
        except StaleElementReferenceException:
            return False

    try:
        return WebDriverWait(driver, 10).until(has_expected_value)
    except TimeoutException:
        return False


def debug_wikidata_page(driver, entity_path, lngid):
    """Dump the state of one edit page. Read-only: nothing is written to TMDB."""
    driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')
    time.sleep(3)

    print(f'url          : {driver.current_url}')
    print(f'title        : {driver.title}')
    print(f'logged out   : {is_logged_out(driver)}')
    print(f'page 404     : {is_page_not_found(driver)}')
    print(f'locked       : {is_wikidata_id_locked(driver)}')
    print(f'save buttons : {len(driver.find_elements(By.XPATH, SAVE_BUTTON_XPATH))}')

    fields = driver.find_elements(By.NAME, 'wikidata_id')
    print(f'wikidata_id inputs: {len(fields)}')
    for position, field in enumerate(fields):
        print(f'  [{position}] {describe_field_state(field)} value={field.get_attribute("value")!r}')
        print(f'       {field.get_attribute("outerHTML")[:400]}')

    for status in driver.find_elements(By.CSS_SELECTOR, '#wikidata_id_status'):
        print(f'  status: {status.get_attribute("outerHTML")[:300]}')


def set_wikidata_id(driver, entity_path, lngid, strwikidataid, timeout_message):
    driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')
    intpagefound = True
    try:
        if is_page_not_found(driver):
            print("Element 'Page 404' exists on the page.")
            intpagefound = False
        else:
            print("Element 'Page 404' does not exist on the page.")
    except NoSuchElementException:
        print("Element 'Page 404' does not exist on the page.")

    if intpagefound:
        xpath = "//button[span[contains(@class, 'glyphicons_v2') and contains(@class, 'plus') and contains(@class, 'svg')] and contains(., 'Create Translation')]"
        try:
            button = driver.find_element(By.XPATH, xpath)
            button.click()
            print("Button 'Create translation' clicked successfully.")
            time.sleep(2)
            driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')
        except Exception:
            print("Button 'Create translation' not found, translation already exists.")

        wikidata_id_field = get_wikidata_id_field(driver, lngid, timeout_message)
        if wikidata_id_field is None:
            if is_logged_out(driver):
                print('TMDB session lost: run the login cell again before resuming.')
            return False

        if is_wikidata_id_locked(driver):
            print("Element 'Locked' exists on the page.")
            return False

        if not is_field_writable(wikidata_id_field):
            print(
                f'wikidata_id field is not editable for {entity_path} {lngid} '
                f'({describe_field_state(wikidata_id_field)}). Skipping.'
            )
            return False

        if not write_wikidata_id(wikidata_id_field, strwikidataid):
            print(f'Skipping {entity_path} {lngid}.')
            return False

        save_button = driver.find_element(By.XPATH, SAVE_BUTTON_XPATH)
        save_button.click()

        if not confirm_wikidata_id(driver, strwikidataid):
            print(f'Save not confirmed for {entity_path} {lngid}: {strwikidataid} was not applied.')
            return False

        print(f'{entity_path} {lngid} updated with {strwikidataid}.')
        return True

    return False


def clear_wikidata_id(driver, entity_path, lngid):
    driver.get(f'https://www.themoviedb.org/{entity_path}/{lngid}/edit?active_nav_item=external_ids')

    if is_page_not_found(driver):
        print("Element 'Page 404' exists on the page. Skipping clear.")
        return False

    wikidata_id_field = get_wikidata_id_field(
        driver, lngid, f'wikidata_id field not found for {entity_path} {{lngid}}. Skipping clear.'
    )
    if wikidata_id_field is None:
        if is_logged_out(driver):
            print('TMDB session lost: run the login cell again before resuming.')
        return False

    if is_wikidata_id_locked(driver):
        print("Element 'Locked' exists on the page. Skipping clear.")
        return False

    if not is_field_writable(wikidata_id_field):
        print(
            f'wikidata_id field is not editable for {entity_path} {lngid} '
            f'({describe_field_state(wikidata_id_field)}). Skipping clear.'
        )
        return False

    if not write_wikidata_id(wikidata_id_field):
        print(f'Skipping clear for {entity_path} {lngid}.')
        return False

    save_button = driver.find_element(By.XPATH, SAVE_BUTTON_XPATH)
    save_button.click()

    if not confirm_wikidata_id(driver, ''):
        print(f'Clear not confirmed for {entity_path} {lngid}.')
        return False

    return True


def process_dataset(driver, dataset):
    latest_csv = get_latest_csv(dataset['local_folder'], dataset['local_file_basename'])
    data = pd.read_csv(str(latest_csv), sep=';', quotechar='"')
    print(data.shape)

    store_key = dataset['store_key']
    start_value = 0
    print(f'{store_key} = {start_value}')
    processed_count = 0

    if dataset.get('name') == 'persons' and not dataset.get('intgoingdown', True):
        rows = data.iloc[::-1].iterrows()
        comparator = lambda current_id, last_id: current_id < last_id
    else:
        rows = data.iterrows()
        comparator = lambda current_id, last_id: current_id > last_id

    for index, row in rows:
        print(f'Index: {index} ; processed: {processed_count}')

        if row[dataset['id_column']]:
            lngid = row[dataset['id_column']]
            if row['ID_WIKIDATA']:
                strwikidataid = row['ID_WIKIDATA']
                strtmdbidtoerase = row[dataset['erase_column']]
                if comparator(lngid, start_value):
                    if pd.isna(strtmdbidtoerase):
                        print('strtmdbidtoerase is NaN')
                    else:
                        print('strtmdbidtoerase is not NaN')
                        lngtmdbidtoerase = int(strtmdbidtoerase)
                        clear_wikidata_id(driver, dataset['entity_path'], lngtmdbidtoerase)

                    was_updated = set_wikidata_id(
                        driver,
                        dataset['entity_path'],
                        lngid,
                        strwikidataid,
                        dataset['timeout_message']
                    )
                    if was_updated:
                        append_processed_log(dataset['name'], lngid, strwikidataid)
                    processed_count += 1
                    start_value = lngid
                    set_store_value(store_key, start_value)
                    time.sleep(2)
            else:
                print(dataset['missing_wikidata_message'])
        else:
            print(dataset['missing_entity_message'])


In [6]:
for dataset in DATASETS:
    download_new_csv_files(dataset['remote_folder'], dataset['local_folder'])

driver = init_driver()
login_tmdb(driver)


SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-movie-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-movie-fix
Downloaded 0 new file(s), skipped 50 existing file(s)
SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-serie-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-serie-fix
Downloaded 0 new file(s), skipped 50 existing file(s)
SFTP folder: /home/debian/docker/selenium-tmdb/wikidata-id-person-fix
Local folder: C:\Users\vaugo\Code\selenium-tmdb\data\wikidata-id-person-fix
Downloaded 0 new file(s), skipped 50 existing file(s)


In [7]:
# Diagnostic (read-only, no write to TMDB): inspect one edit page to understand
# why the wikidata_id field is refused. Run it after the login cell.
debug_wikidata_page(driver, 'movie', 320496)


url          : https://www.themoviedb.org/movie/320496-thaaliya-bhagya/edit?active_nav_item=external_ids
title        : Edit Thaaliya Bhagya — The Movie Database (TMDB)
logged out   : False
page 404     : False
locked       : False
save buttons : 1
wikidata_id inputs: 1
  [0] displayed=True enabled=True readonly=None value=''
       <input dir="auto" id="wikidata_id" class="k-input k-input-solid k-input-md k-rounded-md k-input-inner" type="text" name="wikidata_id" value="" autocomplete="off" data-role="textbox" aria-disabled="false" inputmode="text" style="width: 100%;">
  status: <span id="wikidata_id_status" class="glyphicons unlocked locked_status mr-0!"></span>


In [8]:
process_dataset(driver, DATASETS[0])
process_dataset(driver, DATASETS[1])
process_dataset(driver, DATASETS[2])


Using latest CSV: data\wikidata-id-movie-fix\wikidata-id-movie-fix-20260723.csv
(162, 8)
lngmovieidstart = 0
Index: 0 ; processed: 0
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 1 ; processed: 1
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 2 ; processed: 2
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
Stored 'lngmovieidstart' (int)
Index: 3 ; processed: 3
strtmdbidtoerase is NaN
Element 'Page 404' does not exist on the page.
Button 'Create translation' not found, translation already exists.
Element 'Locked' exists on the page.
St

In [9]:
# Loop is finished so we display the home page
driver.get("https://www.themoviedb.org/")